In [1]:
from qm.QuantumMachinesManager import QuantumMachinesManager
from qm.octave import *
from qm.octave.octave_manager import ClockMode
from qm.qua import *
import os
import time
import matplotlib.pyplot as plt
from qualang_tools.units import unit
import config

from OPXexperiments import OPXexp
from General_QM_Experiments import General_QM_Exps

from oldConfig import config as initialConfiguration
from Single_Qubit_Experiments import Single_Qubit_Experiments
import json

2024-01-04 16:01:04,083 - qm - INFO     - Starting session: b6eec867-2931-40d6-a57a-279f3c1f0a74


C:\Users\measPC\AppData\Local\Temp\ipykernel_14060\700462696.py:1: DeprecationWarning: 'qm.QuantumMachinesManager.QuantumMachinesManager' is moved as of 1.1.2 and will be removed in 1.2.0. use 'qm.QuantumMachinesManager' instead
  from qm.QuantumMachinesManager import QuantumMachinesManager


# Configuration 

Octave outputs from configuration. We have 4 elements: a voltage source, a qubit drive, a resonator drive, and a multi qubit drive.

* The qubit drive is RF1

* The resonator drive is RF2

* The multiqubit drive is RF3 (note that this hasn't been setup)

* the voltage source comes from the OPX (note that this hasn't been setup)

In [2]:
#Experiment config (stores parameters for the experiment we are doing)

with open("experiments.json", "r") as f:
    experiment_config = json.load(f)

# Resonator Spectroscopy

## Initial Parameters

In [3]:
nanoTest = Single_Qubit_Experiments()

#Set LO Frequency of resonator
LOFrequency = 7.0e9
nanoTest.changeLOfrequency('resonator', LOFrequency)

#Pulling the experiment from the experiment_config for calibration parameters
experiment_name = 'resspec'
experiment_params = experiment_config[experiment_name]

"""Resonator spectroscopy parameters"""

#IF parameters from the experiment_config (set there, too)
minimum_frequency = experiment_params['IF frequency range (Hz)'][0]
maximum_frequency = experiment_params['IF frequency range (Hz)'][1]
frequency_interval = experiment_params["frequency intervals (Hz)"]

#Number of frequencies we are sweeping through in resonator spectroscopy
numberOfPoints = (int)((maximum_frequency - minimum_frequency)/frequency_interval)



### Demodulation parameters

In [4]:
time_of_flight = 200
smearing = 40


nanoTest.changeElement("resonator", "smearing", smearing)
nanoTest.changeElement("resonator", "time_of_flight", time_of_flight)


nanoTest.OPX_config["elements"]["resonator"]

{'mixInputs': {'I': ('con1', 3),
  'Q': ('con1', 4),
  'lo_frequency': 7000000000.0,
  'mixer': 'octave_octave1_2'},
 'intermediate_frequency': 200000000.0,
 'operations': {'CW': 'CW',
  'saturation': 'saturation_pulse',
  'long_readout': 'long_readout_pulse',
  'readout': 'readout_pulse',
  'test': 'test_readout_pulse'},
 'outputs': {'out1': ('con1', 1), 'out2': ('con1', 2)},
 'time_of_flight': 200,
 'smearing': 40,
 'digitalInputs': {'Switch': {'port': ('con1', 3), 'delay': 0, 'buffer': 0}}}

## Starting Quantum Machine

### Calibration Start

Before calibrating, make sure to connect RF1 to RF1in and RF2 to RF2in. afterwards return to original setup

In [ ]:
"""Start Quantum Machines with calibration"""

#Temporary measure to ignore deprecation warnings
# import warnings
# warnings.filterwarnings("ignore", category=DeprecationWarning) 

#Calibration step

#Calibrate the resonator using the calibration_params.json file
nanoTest.editCalibrationParams("resonator", "calibration_params.json", 
    LOFrequencies= [LOFrequency for n in range(numberOfPoints)],
    IFFrequencies= [frequency for frequency in range(int(minimum_frequency), int(maximum_frequency), int(frequency_interval))],
    # LOFrequencies= [7.0e9],
    # IFFrequencies= [200e6],
)
# nanoTest.qm.octave.set_rf_output_gain(element = "resonator", gain_in_db = 20)
# nanoTest.qm.octave.calibrate_element("resonator", [(7.2e9, 15e6)])
nanoTest.start_QM_octave(calibration = True)

In [ ]:
#Test continuous pulse for 60 seconds
with program() as hello_octave:
    with infinite_loop_():
        el = "resonator"
        play("CW", el)

job = nanoTest.qm.execute(hello_octave)
time.sleep(30)  # The program will run for 1 minute
job.halt()


### No calibration Start

In [5]:
"""Start Quantum Machine Without Calibration"""

nanoTest.start_QM_octave(calibration = False)

2024-01-04 14:39:13,434 - qm - INFO     - Octave "octave1" Health check passed, current temperature 59
2024-01-04 14:39:13,438 - qm - INFO     - Performing health check
2024-01-04 14:39:13,450 - qm - INFO     - Health check passed


d:\anaconda3\lib\site-packages\qm\octave\octave_config.py:342: DeprecationWarning: Setting port mapping was moved to the config, please move your mapping there
  conns = self.get_opx_octave_port_mapping()
d:\anaconda3\lib\site-packages\qm\octave\octave_config.py:342: DeprecationWarning: Setting port mapping was moved to the config, please move your mapping there
  conns = self.get_opx_octave_port_mapping()
d:\anaconda3\lib\site-packages\qm\octave\octave_config.py:342: DeprecationWarning: Setting port mapping was moved to the config, please move your mapping there
  conns = self.get_opx_octave_port_mapping()


## Running Experiment

In [6]:
#Change the output gain
elementsAndGains = {
    "resonator" : 0
}
#Run our experiment
results = nanoTest.RUN_experiment('resspec', experiment_config, elementsAndGains= elementsAndGains)

2024-01-04 14:39:21,663 - qm - INFO     - Performing health check
2024-01-04 14:39:21,672 - qm - INFO     - Health check passed


d:\anaconda3\lib\site-packages\qm\octave\octave_config.py:342: DeprecationWarning: Setting port mapping was moved to the config, please move your mapping there
  conns = self.get_opx_octave_port_mapping()
d:\anaconda3\lib\site-packages\qm\octave\octave_config.py:342: DeprecationWarning: Setting port mapping was moved to the config, please move your mapping there
  conns = self.get_opx_octave_port_mapping()
d:\anaconda3\lib\site-packages\qm\octave\octave_config.py:342: DeprecationWarning: Setting port mapping was moved to the config, please move your mapping there
  conns = self.get_opx_octave_port_mapping()


2024-01-04 14:39:27,622 - qm - INFO     - Sending program to QOP for compilation
2024-01-04 14:39:27,809 - qm - INFO     - Executing program


## Data Collection and plotting ##

TO BE IMPLEMENTED 

# Qubit Spectroscopy

## Initial Parameters ##

### Experiment Parameters

In [11]:
nanoTest = Single_Qubit_Experiments()

#Set LO Frequency and IF frequency of resonator
LOFrequency = 7.0e9
nanoTest.changeLOfrequency('resonator', LOFrequency)
IFFrequency = 200e6
nanoTest.changeIFfrequency('resonator', IFFrequency)

#Set Qubit LO frequency
LOFrequency = 6.2e9
nanoTest.changeLOfrequency('qubit', LOFrequency)

#Pulling the experiment from the experiment_config for calibration parameters
experiment_name = 'qubitspec'
experiment_params = experiment_config[experiment_name]

"""Resonator spectroscopy parameters"""

#IF parameters from the experiment_config (set there, too)
minimum_frequency = experiment_params['IF frequency range (Hz)'][0]
maximum_frequency = experiment_params['IF frequency range (Hz)'][1]
frequency_interval = experiment_params["frequency intervals (Hz)"]


#Number of frequencies we are sweeping through in qubit spectroscopy
numberOfPoints = (int)((maximum_frequency - minimum_frequency)/frequency_interval)



### Change Waveform Parameters

In [12]:
"""
We use 2 pulses in this experiment: 

A saturation pulse which puts the qubit into a 50/50 mixture

A readout pulse
"""

saturation_voltage = 0.4
readout_voltage = 0.6

nanoTest.changeSample(saturation_voltage, "saturation_wf")
nanoTest.changeSample(readout_voltage, 'readout_wf')


{'type': 'constant', 'sample': 0.6}

## Starting Quantum Machine

### Starting with calibration

TO BE IMPLEMENTED

### Starting Without Calibration

In [13]:
nanoTest.start_QM_octave(calibration = False)

2024-01-04 15:04:41,770 - qm - INFO     - Performing health check
2024-01-04 15:04:41,781 - qm - INFO     - Health check passed


d:\anaconda3\lib\site-packages\qm\octave\octave_config.py:342: DeprecationWarning: Setting port mapping was moved to the config, please move your mapping there
  conns = self.get_opx_octave_port_mapping()
d:\anaconda3\lib\site-packages\qm\octave\octave_config.py:342: DeprecationWarning: Setting port mapping was moved to the config, please move your mapping there
  conns = self.get_opx_octave_port_mapping()
d:\anaconda3\lib\site-packages\qm\octave\octave_config.py:342: DeprecationWarning: Setting port mapping was moved to the config, please move your mapping there
  conns = self.get_opx_octave_port_mapping()


## Running Experiment

In [19]:
#Change the output gain
elementsAndGains = {
    "resonator" : 0,
    'qubit' : 0
}
#Run our experiment
results = nanoTest.RUN_experiment(experiment_name, experiment_config, elementsAndGains= elementsAndGains)

2024-01-04 15:06:29,117 - qm - INFO     - Performing health check
2024-01-04 15:06:29,126 - qm - INFO     - Health check passed


d:\anaconda3\lib\site-packages\qm\octave\octave_config.py:342: DeprecationWarning: Setting port mapping was moved to the config, please move your mapping there
  conns = self.get_opx_octave_port_mapping()
d:\anaconda3\lib\site-packages\qm\octave\octave_config.py:342: DeprecationWarning: Setting port mapping was moved to the config, please move your mapping there
  conns = self.get_opx_octave_port_mapping()
d:\anaconda3\lib\site-packages\qm\octave\octave_config.py:342: DeprecationWarning: Setting port mapping was moved to the config, please move your mapping there
  conns = self.get_opx_octave_port_mapping()


2024-01-04 15:06:35,536 - qm - INFO     - Sending program to QOP for compilation
2024-01-04 15:06:35,764 - qm - INFO     - Executing program


## Data Collection and Plotting

TO BE IMPLEMENTED

In [9]:
print(results["I"])
print(results["Q"])

[-5.4672455e-11]
[-5.4672455e-11]


# Rabi Experiment

## Initial Parameters

### Experiment Parameters

In [4]:
nanoTest = Single_Qubit_Experiments()

#Set LO Frequency and IF frequency of resonator
LOFrequency = 7.0e9
nanoTest.changeLOfrequency('resonator', LOFrequency)
IFFrequency = 200e6
nanoTest.changeIFfrequency('resonator', IFFrequency)

#Set Qubit LO frequency and IF frequency
LOFrequency = 6.2e9
nanoTest.changeLOfrequency('qubit', LOFrequency)
IFFrequency = 370e6
nanoTest.changeIFfrequency('qubit', IFFrequency)

#Pulling the experiment from the experiment_config for calibration parameters
experiment_name = 'rabi1d'
experiment_params = experiment_config[experiment_name]

"""Resonator spectroscopy parameters"""

#time interval parameters from the experiment_config (set there, too)
minimum_time = experiment_params['time range (ns)'][0]
maximum_time = experiment_params['time range (ns)'][1]
time_interval = experiment_params["time interval (ns)"]


#Number of times we are sweeping through in rabi experiment
numberOfPoints = (int)((maximum_time - minimum_time)/time_interval)



### Change Waveform Parameters

In [5]:
"""
We use 2 pulses in this experiment: 

A saturation pulse which puts the qubit into a 50/50 mixture

A readout pulse
"""

saturation_voltage = 0.4
readout_voltage = 0.6

nanoTest.changeSample(saturation_voltage, "saturation_wf")
nanoTest.changeSample(readout_voltage, 'readout_wf')


{'type': 'constant', 'sample': 0.6}

## Starting Quantum Machine

### Starting with calibration

TO BE IMPLEMENTED

### Starting Without Calibration

In [6]:
nanoTest.start_QM_octave(calibration = False)

2024-01-04 15:17:22,097 - qm - INFO     - Octave "octave1" Health check passed, current temperature 59
2024-01-04 15:17:22,099 - qm - INFO     - Performing health check
2024-01-04 15:17:22,109 - qm - INFO     - Health check passed


d:\anaconda3\lib\site-packages\qm\octave\octave_config.py:342: DeprecationWarning: Setting port mapping was moved to the config, please move your mapping there
  conns = self.get_opx_octave_port_mapping()
d:\anaconda3\lib\site-packages\qm\octave\octave_config.py:342: DeprecationWarning: Setting port mapping was moved to the config, please move your mapping there
  conns = self.get_opx_octave_port_mapping()
d:\anaconda3\lib\site-packages\qm\octave\octave_config.py:342: DeprecationWarning: Setting port mapping was moved to the config, please move your mapping there
  conns = self.get_opx_octave_port_mapping()


## Running Experiment

In [7]:
#Change the output gain
elementsAndGains = {
    "resonator" : 0,
    'qubit' : 0
}
#Run our experiment
results = nanoTest.RUN_experiment(experiment_name, experiment_config, elementsAndGains= elementsAndGains)

2024-01-04 15:17:28,558 - qm - INFO     - Performing health check
2024-01-04 15:17:28,570 - qm - INFO     - Health check passed


d:\anaconda3\lib\site-packages\qm\octave\octave_config.py:342: DeprecationWarning: Setting port mapping was moved to the config, please move your mapping there
  conns = self.get_opx_octave_port_mapping()
d:\anaconda3\lib\site-packages\qm\octave\octave_config.py:342: DeprecationWarning: Setting port mapping was moved to the config, please move your mapping there
  conns = self.get_opx_octave_port_mapping()
d:\anaconda3\lib\site-packages\qm\octave\octave_config.py:342: DeprecationWarning: Setting port mapping was moved to the config, please move your mapping there
  conns = self.get_opx_octave_port_mapping()


2024-01-04 15:17:34,722 - qm - INFO     - Sending program to QOP for compilation
2024-01-04 15:17:34,942 - qm - INFO     - Executing program


## Data Collection and Plotting

TO BE IMPLEMENTED

In [ ]:
print(results["I"])
print(results["Q"])

[-5.4672455e-11]
[-5.4672455e-11]


# T1 Experiment

## Initial Parameters

### Experiment Parameters

In [3]:
nanoTest = Single_Qubit_Experiments()

#Set LO Frequency and IF frequency of resonator
LOFrequency = 7.0e9
nanoTest.changeLOfrequency('resonator', LOFrequency)
IFFrequency = 200e6
nanoTest.changeIFfrequency('resonator', IFFrequency)

#Set Qubit LO frequency and IF frequency
LOFrequency = 6.2e9
nanoTest.changeLOfrequency('qubit', LOFrequency)
IFFrequency = 370e6
nanoTest.changeIFfrequency('qubit', IFFrequency)

#Pulling the experiment from the experiment_config for calibration parameters
experiment_name = 'T1'
experiment_params = experiment_config[experiment_name]

"""Resonator spectroscopy parameters"""

#time interval parameters from the experiment_config (set there, too)
minimum_time = experiment_params['time range (ns)'][0]
maximum_time = experiment_params['time range (ns)'][1]
time_interval = experiment_params["time interval (ns)"]


#Number of times we are sweeping through in rabi experiment
numberOfPoints = (int)((maximum_time - minimum_time)/time_interval)



### Change Waveform Parameters

In [4]:
"""
We use 2 pulses in this experiment: 

A saturation pulse which puts the qubit into a 50/50 mixture

A readout pulse
"""

saturation_voltage = 0.4
readout_voltage = 0.6

nanoTest.changeSample(saturation_voltage, "saturation_wf")
nanoTest.changeSample(readout_voltage, 'readout_wf')


{'type': 'constant', 'sample': 0.6}

## Starting Quantum Machine

### Starting with calibration

TO BE IMPLEMENTED

### Starting Without Calibration

In [5]:
nanoTest.start_QM_octave(calibration = False)

2024-01-04 16:01:19,230 - qm - INFO     - Octave "octave1" Health check passed, current temperature 59
2024-01-04 16:01:19,232 - qm - INFO     - Performing health check
2024-01-04 16:01:19,243 - qm - INFO     - Health check passed


d:\anaconda3\lib\site-packages\qm\octave\octave_config.py:342: DeprecationWarning: Setting port mapping was moved to the config, please move your mapping there
  conns = self.get_opx_octave_port_mapping()
d:\anaconda3\lib\site-packages\qm\octave\octave_config.py:342: DeprecationWarning: Setting port mapping was moved to the config, please move your mapping there
  conns = self.get_opx_octave_port_mapping()
d:\anaconda3\lib\site-packages\qm\octave\octave_config.py:342: DeprecationWarning: Setting port mapping was moved to the config, please move your mapping there
  conns = self.get_opx_octave_port_mapping()


## Running Experiment

In [6]:
#Change the output gain
elementsAndGains = {
    "resonator" : 0,
    'qubit' : 0
}
#Run our experiment
results = nanoTest.RUN_experiment(experiment_name, experiment_config, elementsAndGains= elementsAndGains)

2024-01-04 16:01:28,206 - qm - INFO     - Performing health check
2024-01-04 16:01:28,216 - qm - INFO     - Health check passed


d:\anaconda3\lib\site-packages\qm\octave\octave_config.py:342: DeprecationWarning: Setting port mapping was moved to the config, please move your mapping there
  conns = self.get_opx_octave_port_mapping()
d:\anaconda3\lib\site-packages\qm\octave\octave_config.py:342: DeprecationWarning: Setting port mapping was moved to the config, please move your mapping there
  conns = self.get_opx_octave_port_mapping()
d:\anaconda3\lib\site-packages\qm\octave\octave_config.py:342: DeprecationWarning: Setting port mapping was moved to the config, please move your mapping there
  conns = self.get_opx_octave_port_mapping()


2024-01-04 16:01:34,462 - qm - INFO     - Sending program to QOP for compilation
2024-01-04 16:01:34,692 - qm - INFO     - Executing program


## Data Collection and Plotting

TO BE IMPLEMENTED

In [ ]:
print(results["I"])
print(results["Q"])

[-5.4672455e-11]
[-5.4672455e-11]
